## 1) Inspiration

Having personally been in a large car crash last summer, I want to understand how to predict the severity of a car crash under the conditions of the crash such as the weather or the speed limits of the relevant roadways. Car crashes are also the leading cause of death for young people below the age of 22 and the 2nd most common cause of death for people from ages 22 to 67. Analysis of this data could provide guidelines that lower car crashes, which I would greatly desire. 

## 2) Stakeholders

The stakeholders of this project would include anyone who drives a motor vehicle daily in Chicago; even perfect drivers might become involved in collision due to the actions of another driver or become delayed in traffic as a result of a crash. City officials of Chicago who have to decide where to post traffic police, speed cameras, and speed limit signs, as well as determine what those speed limits should be, will benefit from this. Additionally, this data can help tourists decide when the safest times to visit Chicago are, and help paramedics determine which types of streets/periods of weather produce the most dangerous crashes.

## 3) Task and Metrics

This will be a regression problem. The variable will be a custom variable designed called "injury score" which sums the product of each injury in a collision with its associated severity (from no injury to death).

I will evaluate the prediction performance of my models based on the standard regression performance metrics of root mean squared error (RMSE) to expresses prediction error in the same units as the injury score, making the model’s typical error magnitude easy to interpret practically, mean absolute error (MAE) to measure the performance of the model outside major errors, along with the coefficient of determination (R^2) to assess overall goodness of fit.


## 4) Data

- Identify and describe your data source(s). Include the links/citations, so the readers can access the data.
- Describe **(no code or code output here)** the following:
    - How many observations and variables does the dataset have?
    - What does each observation and variable represent?
    - How many variables are numeric and how many variables are categorical?
    - Which variables are the predictors and which variable is the response?
 
If the data needed to be cleaned for your analysis, **briefly** describe the steps you took to clean the data. (Data cleaning was last quarter's focus, so it should not be a major part of this report.) **If the data was clean, state "cleaning was not necessary."**
 
**(2 points)**

The data is comprised of three parts - crash data, beat population data and weather data. 

There are approximately 1.008 million observations in the crash data; 18 of the variables are numeric, and the remaining 30 variables are categorical. During data cleaning/reformatting, a new variable called "Injury_Score" was added using the following formula; 
$$
\begin{align}
\text{Injury\_Score} &= \text{Injuries\_Fatal} \times 5 + \text{Injuries\_Incapacitating} \times 3 \\
&\quad + \text{ Injuries\_Non\_Incapacitating} \times 2 + (\text{Injuries\_Reported\_Not\_Evident} + \text{Injuries\_Unknown}) \times 1 \\
&\quad + \text{ Injuries\_No\_Indication} \times 0.5
\end{align}
$$

2171 crashes where any of the variables detailing the number of injuries at a particular severity were missing were removed, roughly 0.2% of all other crashes. In addition, the population of each crash's beat was inner joined into the crash dataset, with a final total of 853k entries. The hour of the crash was also transformed manually into a categorical variable 'time_of_day'. 

There are 233,414 observations in the weather data from the Chicago Midway airport. 9 of the variables are numeric and 2 are categorical. This is less than 10 numeric variables, but since this is supplementary data that is meant to be added onto the crash data set, combined with the crash data variables it will be more than 10 in total. There are some missing hours from the dataset, which required imputing values from adjacent days at a similar time. Weather reports were merged into the crash data with each crash having weather information from the closest available weather report within 2 hours, with crashes removed if no weather report within that time period could be found.

The predictor variables used are the number of vehicles involved in the crash, population of the beat where the crash occurred, the time of day, speed limit, precipitation rate, lighting, temperature and windspeed, and the response variable will be the crash injury score. The other variables judged as irrelevant to the severity of crash injuries (such as how the air pressure or how many photos were taken) or being categorical variables with far too many categories for efficient regression analysis.

Source 1: https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if/about_data

Source 2: https://www.wunderground.com/history/daily/us/il/chicago/KMDW (datascraped by me; script using for scraping available on request)

Source 3: https://www.kaggle.com/datasets/robertyu02/cpd-police-beat-demographics?resource=download

## 5) Prediction

In [5]:
#| echo: false
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
crash = pd.read_csv('Traffic_Crashes_-_Crashes_20251109.csv')
injury_columns = [
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION',
    'INJURIES_UNKNOWN'
]
negative_value_check = {}
for column in injury_columns:
    has_negative = (crash[column]<0).any()
    negative_value_check[column] = has_negative
check_results = pd.Series(negative_value_check)

empty_value_count = {}
for column in injury_columns:
    count = crash[column].isnull().sum()
    empty_value_count[column] = count
check_results = pd.Series(empty_value_count)
crash_injury_filtered = crash.dropna(subset=injury_columns)
crash_injury_filtered = crash_injury_filtered.dropna(subset = ['BEAT_OF_OCCURRENCE'])
beat_data = pd.read_csv('beatpop.txt', sep='\s+').drop(columns = ['area'])
crash_injury_filtered['BEAT_OF_OCCURRENCE'] = crash_injury_filtered['BEAT_OF_OCCURRENCE'].astype(int)
crash_merged = crash_injury_filtered.merge(beat_data, left_on='BEAT_OF_OCCURRENCE', right_on='beat_number', how='inner')
#weather cleaning and merge into crash database
weather = pd.read_csv('KMDW.csv')
weather['DateTime_String'] = weather['Date'].astype(str) + ' ' + weather['Time'].astype(str)
weather['US_Time'] = pd.to_datetime(weather['DateTime_String'], format = '%Y-%m-%d %I:%M %p')
crash_merged['US_Time'] = pd.to_datetime(
    crash_merged['CRASH_DATE'], 
    format='%m/%d/%Y %I:%M:%S %p'
)
weather = weather.sort_values(by = 'US_Time', ascending = True)
crash_merged_sorted = crash_merged.sort_values(by = 'US_Time', ascending = True)
from datetime import timedelta
crash_merged_with_weather = pd.merge_asof(
    crash_merged_sorted, 
    weather, 
    on='US_Time', 
    direction='nearest', 
    tolerance=timedelta(minutes=120)
)
crash_merged_with_weather['INJURY_SCORE'] = crash_merged_with_weather['INJURIES_FATAL']*5 + crash_merged_with_weather['INJURIES_INCAPACITATING']*3 + crash_merged_with_weather['INJURIES_NON_INCAPACITATING']*2 + \
crash_merged_with_weather['INJURIES_REPORTED_NOT_EVIDENT']*1 + crash_merged_with_weather['INJURIES_UNKNOWN']*1 +crash_merged_with_weather['INJURIES_NO_INDICATION']*0.5
def time_of_day(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

crash_merged_with_weather['TIME_OF_DAY'] = crash_merged_with_weather['CRASH_HOUR'].apply(time_of_day)
predictors = ['US_Time', 'NUM_UNITS', 'population', 'TIME_OF_DAY', 'POSTED_SPEED_LIMIT', 'Precipitation Rate in mm', 'LIGHTING_CONDITION', 'Temperature_C', 'Speed in km/h']
crash_predictors_with_time = crash_merged_with_weather[predictors]
crash_predictors_with_time = crash_predictors_with_time.sort_values('US_Time')

for col in ['Precipitation Rate in mm', 'Temperature_C', 'Speed in km/h']:
    crash_predictors_with_time[col] = crash_predictors_with_time[col].ffill()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

predictors = ['NUM_UNITS', 'TIME_OF_DAY', 'POSTED_SPEED_LIMIT', 'Precipitation Rate in mm', 'population', 'LIGHTING_CONDITION', 'Temperature_C', 'Speed in km/h']


X = crash_predictors_with_time[predictors]
X = X.rename(columns={'NUM_UNITS': '#_of_vehicles', 'population': 'beat_population', 'Speed in km/h':'Wind speed(km/h)'})

X_ohe = pd.get_dummies(X)

y = crash_merged_with_weather['INJURY_SCORE']

X_train, X_test, y_train, y_test = train_test_split(X_ohe, y, test_size=0.1, random_state=1)

from sklearn.metrics import r2_score

results = []
col_groups = []
i = 0

while i < len(X_ohe.columns):
    col = X_ohe.columns[i]
    # Check if this column is part of an OHE group (contains '_')
    base_name = col.rsplit('_', 1)[0]
    group = [c for c in X_ohe.columns[i:] if c.startswith(base_name + '_')]
    
    if len(group) > 1:
        col_groups.append(group)
        i += len(group)
    else:
        col_groups.append([col])
        i += 1

current_cols = []
for group in col_groups:
    current_cols.extend(group)
    
    model = LinearRegression()
    model.fit(X_train[current_cols], y_train)
    
    train_pred = model.predict(X_train[current_cols])
    test_pred = model.predict(X_test[current_cols])
    
    results.append({
        'Predictors': ', '.join(current_cols),
        'Train_RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train_MAE': mean_absolute_error(y_train, train_pred),
        'Test_MAE': mean_absolute_error(y_test, test_pred),
        'Train_R2': r2_score(y_train, train_pred),
        'Test_R2': r2_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)

results_df


,Predictors,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_R2,Test_R2
0,#_of_vehicles,1.113769,1.115746,0.687197,0.684028,0.027860,0.025276
1,"#_of_vehicles, POSTED_SPEED_LIMIT",1.107883,1.110518,0.680936,0.678076,0.038107,0.034390
2,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm",1.107881,1.110508,0.680931,0.678061,0.038111,0.034407
3,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population",1.106676,1.109251,0.679458,0.676353,0.040202,0.036591
4,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C",1.106335,1.109047,0.679058,0.676032,0.040794,0.036946
5,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C, Wind speed(km/h)",1.106334,1.109050,0.679062,0.676037,0.040795,0.036942
6,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C, Wind speed(km/h), TIME_OF_DAY_Afternoon, TIME_OF_DAY_Evening, TIME_OF_DAY_Morning, TIME_OF_DAY_Night",1.105557,1.108538,0.677606,0.674808,0.042142,0.037830
7,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C, Wind speed(km/h), TIME_OF_DAY_Afternoon, TIME_OF_DAY_Evening, TIME_OF_DAY_Morning, TIME_OF_DAY_Night, LIGHTING_CONDITION_DARKNESS, LIGHTING_CONDITION_DARKNESS, LIGHTED ROAD, LIGHTING_CONDITION_DAWN, LIGHTING_CONDITION_DAYLIGHT, LIGHTING_CONDITION_DUSK, LIGHTING_CONDITION_UNKNOWN",1.101669,1.104611,0.673157,0.670252,0.048866,0.044635


In [6]:
#| echo: false
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import RidgeCV

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)


alphas = np.logspace(-3, 3, 100)
ridge = RidgeCV(alphas=alphas, cv=10)
ridge.fit(X_train_scaled, y_train)


train_pred = ridge.predict(X_train_scaled)
test_pred = ridge.predict(X_test_scaled)

# Top 3 most important predictors by absolute coefficient
feature_names = poly.get_feature_names_out(X_train.columns)
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': ridge.coef_
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
top3 = coef_df.nlargest(3, 'Abs_Coefficient')[['Feature', 'Coefficient']].reset_index(drop=True)
top3.index += 1
top3

,Feature,Coefficient
1,#_of_vehicles POSTED_SPEED_LIMIT,0.308838
2,#_of_vehicles TIME_OF_DAY_Night,-0.169075
3,#_of_vehicles TIME_OF_DAY_Afternoon,0.101684


The number of units involved in the crash was chosen as the starting predictor, as common sense indicated it would have the most influential impact on the severity of the overall crash; multiple cars crashing are obviously going to correlate to a greater number of injuries and more severe injuries than a single car crashing into poles. The most informative predictors appear to be the # of units involved, the posted speed limit of the location of the crash and adding the lighting condition categorical variable. The number of units involved in the crash unsurprisingly plays the most important role in determining the deadliness of a crash; as the baseline predictor it is able to achieve a test R² of 0.028 and a training R² of 0.025, a test RMSE of 1.115 and a training RMSE of 1.114, and 0.68 testing/training MAE values. There is minimal overfitting in the baseline, with the lowest difference between test and training RMSE values compared to the other rows, and the addition of other variables into the linear regression model provides diminishing returns compared to the performance values yielded from the number of units predictor.

Adding the posted speed limit of the location of the crash produced the largest improvements in all test and training metrics compared to the other variables; test and training R² improved by 0.009 and 0.01, test and training RMSE decreased by 0.0052 and 0.0058 respectively, and test and training MAE decreased by 0.0059 and 0.0062, respectively. Adding the lighting condition variables produced the second largest improvements in all metrics, with RMSE metrics decreasing by 0.0039, MAE metrics decreasing by 0.0045 and R² metrics increasing by 0.0068 in both testing and training cases.

The least informative predictors appeared to be precipitation rate, windspeed and temperature. They produced essentially negligible changes in performance metrics both during testing and training, with windspeed actually increasing test MAE metrics and decreasing R² values by a slight amount. 

The trained and tuned performance using a ten-fold Ridge cross validation algorithm yielded a hyperparameter of 497.7024, training and test RMSEs of 1.0958 and 1.0986 respectively, training and test MAEs of 0.6696 and 0.6666 respectively with a training and testing coefficient of determination of 0.0590 and 0.0551, respectively. 

The top 3 coefficients were the number of vehicles times the posted speed limit, the number of vehicles times the time of day being night, and the number of vehicles times the time of day being the afternoon. This strongly suggest non-linearities in the underlying relationship between the predictors and the responses. Of the top 15 coefficients by absolute value in the trained and tuned model, the only linear predictor is the posted speed limit value, with an absolute coefficient of 0.09. While the R² values in the tuned Ridge Polynomial model represented a 37% increase from the linear model, the R² values are still undeniably low at around 0.055. Therefore, the non linearities in the data are present but statistically very minor and don't capture the major drivers of what is responsible for higher injury scores.


## 6) Inference

The five predictors chosen were speed limit, time of day, number of vehicles, road lighting conditions and beat population, as they were the variables that had the most impact on the linear regression metrics and each had a interaction coefficient which was among the top 15 coefficients in terms of absolute value within the list of feature coefficients in the Ridge regression model. The interaction terms chosen among these five predictors were chosen from the highest coefficients from the Ridge regression model in terms of absolute value. 

In [7]:
#| echo: false
train = X_train.copy()
train['INJURY_SCORE'] = y_train.values
train = train.rename(columns = {'#_of_vehicles':'num_of_vehicles'})
import statsmodels.formula.api as smf

model = smf.ols(formula = "INJURY_SCORE ~ POSTED_SPEED_LIMIT + TIME_OF_DAY_Afternoon + TIME_OF_DAY_Night + num_of_vehicles \
+ LIGHTING_CONDITION_UNKNOWN + LIGHTING_CONDITION_DAYLIGHT+ beat_population \
+ num_of_vehicles:POSTED_SPEED_LIMIT + I(beat_population**2) + num_of_vehicles:TIME_OF_DAY_Night + \
num_of_vehicles:TIME_OF_DAY_Afternoon + num_of_vehicles:beat_population + \
num_of_vehicles:LIGHTING_CONDITION_UNKNOWN + POSTED_SPEED_LIMIT:LIGHTING_CONDITION_UNKNOWN + \
POSTED_SPEED_LIMIT:TIME_OF_DAY_Night", data=train).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           INJURY_SCORE   R-squared:                       0.055
Model:                            OLS   Adj. R-squared:                  0.055
Method:                 Least Squares   F-statistic:                     3456.
Date:                Mon, 16 Mar 2026   Prob (F-statistic):               0.00
Time:                        22:58:11   Log-Likelihood:            -1.3547e+06
No. Observations:              895590   AIC:                         2.710e+06
Df Residuals:                  895574   BIC:                         2.710e+06
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
=========================================================================================================================
                                                            coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------
Intercept                                                 0.9614      0.028     34.189      0.000       0.906       1.017
TIME_OF_DAY_Afternoon[T.True]                            -0.1873      0.013    -14.121      0.000      -0.213      -0.161
TIME_OF_DAY_Night[T.True]                                 0.2195      0.021     10.465      0.000       0.178       0.261
LIGHTING_CONDITION_UNKNOWN[T.True]                        0.2238      0.040      5.535      0.000       0.145       0.303
LIGHTING_CONDITION_DAYLIGHT[T.True]                      -0.0785      0.003    -25.042      0.000      -0.085      -0.072
POSTED_SPEED_LIMIT                                       -0.0176      0.001    -20.608      0.000      -0.019      -0.016
POSTED_SPEED_LIMIT:LIGHTING_CONDITION_UNKNOWN[T.True]    -0.0088      0.001    -10.921      0.000      -0.010      -0.007
POSTED_SPEED_LIMIT:TIME_OF_DAY_Night[T.True]              0.0130      0.001     22.882      0.000       0.012       0.014
num_of_vehicles                                           0.0589      0.014      4.305      0.000       0.032       0.086
num_of_vehicles:TIME_OF_DAY_Night[T.True]                -0.3076      0.006    -49.907      0.000      -0.320      -0.295
num_of_vehicles:TIME_OF_DAY_Afternoon[T.True]             0.1120      0.006     17.498      0.000       0.099       0.125
num_of_vehicles:LIGHTING_CONDITION_UNKNOWN[T.True]       -0.2130      0.017    -12.330      0.000      -0.247      -0.179
beat_population                                       -1.181e-05   1.19e-06     -9.893      0.000   -1.42e-05   -9.47e-06
num_of_vehicles:POSTED_SPEED_LIMIT                        0.0175      0.000     41.648      0.000       0.017       0.018
I(beat_population ** 2)                                6.393e-10   2.97e-11     21.547      0.000    5.81e-10    6.98e-10
num_of_vehicles:beat_population                        -6.64e-06   4.38e-07    -15.175      0.000    -7.5e-06   -5.78e-06
==============================================================================
Omnibus:                   878763.666   Durbin-Watson:                   2.000
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         97936738.074
Skew:                           4.550   Prob(JB):                         0.00
Kurtosis:                      53.415   Cond. No.                     8.77e+09
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.77e+09. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

TIME_OF_DAY_Afternoon has a coefficient of −0.1873, indicating afternoon collisions are associated with an injury score about 0.19 lower than morning collisions, holding all else constant. This suggests mornings may involve more severe outcomes than afternoons. On the other hand, TIME_OF_DAY_Night has a coefficient of 0.2195. Night collisions are associated with an injury score about 0.22 higher than morning collisions. This makes sense, since reduced visibility and potentially impaired driving increase severity.

Daylight time periods reduce injury score by about 0.08 compared to the reference lighting category, consistent with better visibility leading to less severe outcomes. However, when lighting condition is unknown there is an associated 0.22 increase in injury score. This could reflect missing data in more chaotic or severe incidents where conditions weren't properly recorded. 

A particularly interesting statistic is the coefficient of posted speed limits; each 1 unit increase in speed limit is associated with a 0.018 decrease in injury score. This seems counterintuitive, but likely reflects that higher speed limit roads like highways are better designed with medians, barriers, and fewer pedestrians, while lower speed limit zones have less of these safety measures. 

Each additional vehicle involved increases injury score by about 0.06. This is unsurprising, as more vehicles means more points of impact and more people exposed to injury. On the other hand, an increase in beat_population has a coefficient of −0.00001181. Although this seems contradictory to common sense, areas with more pedestrians may actually encourage drivers to drive safer, leading to lower average injury scores. However,the quadratic transformation of beat population has a positive coefficient of 6.393e-10, which means the relationship between beat population and injury score is U-shaped. Injury score decreases with population up to a point, then starts increasing again at very high population levels, which could reflect the fact that extremely dense urban areas bring back higher severity due to congestion, pedestrian density, or slower responses from emergency services.

When lighting is unknown, the protective effect of higher speed limits is amplified, with each additional unit of speed limit reduces injury score by an extra 0.009 beyond the main effect alone. At night, higher speed limits increase injury severity. This flips the main effect, as while higher speed limits are generally associated with lower injury scores, at night each unit increase in speed limit adds 0.013 to injury score, somewhat offsetting the −0.018 main effect. Likewise, the combination of more vehicles and higher speed limits compounds injury severity. Each additional vehicle at a speed limit one unit higher adds 0.0175 to injury score beyond what either factor contributes alone.

The largest interaction effect is num_of_vehicles × TIME_OF_DAY_Night with a coefficient of -0.31. Multi-vehicle crashes at night are associated with substantially lower injury scores than expected. This could imply that nighttime multi-vehicle incidents tend to be low-speed chain collisions rather than high-impact crashes. Conversely, multi-vehicle crashes in the afternoon increase injury score. Afternoon congestion or faster driving from rush hour conditions may lead to multi-vehicle pileups at higher speeds than at night, leading to greater injury risk.

Under unknown light conditions, multi-vehicle crashes are less severe with a coefficient of -0.2130 from the interaction term. This possibly reflects an artifact in the data, as reports with an unknown lighting condition likely indicate that the report was "lazily" done, and implies the injuries surrounding the crash were not severe enough for the officers make particular note of the area's lighting. Finally, there is a small negative interaction between the number of vehicles and the overall beat population. In more populated beats, additional vehicles contribute less to crash severity because as earlier noted, higher population areas tend to have lower driving speeds and better protective infrastructure.

Every single predictor in the model has a p-value of 0.000, meaning all are statistically significant at the conventional statistical significance level of 0.05 and can be considered reliable. This is largely due to the enormous sample size of 895,590 observations, where even very small effects become statistically detectable. The predictors with the largest absolute t-values and therefore the most reliably estimated effects are num_of_vehicles:TIME_OF_DAY_Night with a t-value of -49.907, num_of_vehicles:POSTED_SPEED_LIMIT with a t-value of 41.648,  and POSTED_SPEED_LIMIT:TIME_OF_DAY_dayligh with a t-value of −25.042. These interaction terms therefore have effects that are very stable and reliable.

However, the R² and adjusted R-squared are both roughly 0.055; this means the model explains about 5.5% of the total variation in injury scores. The vast majority of injury severity variation therefore is driven by factors that are not included in this model, such as driver behavior, seatbelt use, crash angle, emergency response time or the actual speed of collision (as it is not guaranteed that drivers drive at the posted speed limit). The F-statistic of 3456 with an associated p-value of 0.00 means the overall model is statistically significant, but it still explains little of the overall variation in injury scores.

## 7) Recommendation to Stakeholders

The main actions stakeholders should take given this analysis is to avoid nighttime driving, as injury scores are generally higher and the protective effect of well-designed high-speed roads nearly vanishes after dark. Additionally, driving in the afternoon is dangerous in congested areas due to the effects of rush hour congestion causing more daedly crashes. City officials should prioritize nighttime interventions: better lighting, rumble strips, and targeted impaired driving enforcement on higher-speed roads. The speed limit findings indicate infrastructure design matters more than the posted speed limits, since low speed limit zones actually see higher crash severity, likely due to lacking safety infrastructure compared to areas like highways. Investing in pedestrian barriers, crosswalks, and traffic guards in these zones will be more effective than simply lowering speed limits. The parabolic shape of beat population in relation to injury scores indicates that the densest areas and least dense areas require more emergency services than areas of middling density. This is likely because drivers drive safer as population density increases, but only up to a certain point. However this analysis has significant limitations due to the very low variance value of 5.5%, meaning most variations in injury score are explained by unknown factors. To overcome these limitations, collecting crash-level data such as impact speeds, seatbelt wearing rates, EMS response times or driver sobriety could boost the explanative powers of this model substantially.


## 8) Conclusion


This analysis demonstrates that currently available environmental factors, while statistically significant predictors of collision injury severity, capture only a small fraction of the overall variation. The number of vehicles, posted speed limit, lighting conditions, and time of day all showed reliable associations with injury scores, and interaction terms revealed that these factors combine in non-obvious ways, particularly the reversal of speed limit effects at night. On the other hand, weather conditions like precipitation or temperature showed very little relation to injury severity at all. The consistently low R² across both linear and polynomial models of between 0.04 to 0.05 confirms that the major determinants of injury severity lie outside the scope of the available data in as of yet unmeasured factors, such as seatbelt wearing rates or the speed/angle of vehicle collisions. The most impactful policy recommendations center on nighttime safety infrastructure, road design improvements in low-speed zones, more emergency service focus on low and high density population areas, and congestion management during afternoon peak hours. Future analyses should incorporate crash-level variables like actual impact speed, driver sobriety, seatbelt usage rates, and emergency response time to meaningfully improve the explanatory power of the model.